In [1]:
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

# 1. Path configurations - Updated to match your exact file name
image_path = "dogTraining.jpg"
model_path = "model_unquant.tflite"  # Matches your uploaded sidebar file precisely

if not os.path.exists(model_path):
    raise FileNotFoundError(f"'{model_path}' not found! Please check the sidebar explorer.")
if not os.path.exists(image_path):
    raise FileNotFoundError(f"'{image_path}' not found! Please check your file storage.")

# 2. Initialize the TFLite Runtime Interpreter (Pure, native and conflict-free)
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# Get input and output structural tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# 3. Standard Image Preprocessing pipeline matching Teachable Machine dimensions
image = Image.open(image_path).convert("RGB")
size = (224, 224)
image = ImageOps.fit(image, size, Image.Resampling.BILINEAR)

# Normalize the target pixels into vector array [-1, 1]
image_array = np.asarray(image)
normalized_image_array = (image_array.astype(np.float32) / 127.5) - 1
input_data = np.expand_dims(normalized_image_array, axis=0)

# 4. Bind the processed image payload vector into the model input tensor
interpreter.set_tensor(input_details[0]['index'], input_data)

# 5. Run raw native model execution evaluation
interpreter.invoke()

# 6. Retrieve the true structural classification output weights
prediction = interpreter.get_tensor(output_details[0]['index'])

# 7. Map out label indicators configuration
with open("labels.txt", "r") as f:
    class_names = f.readlines()

index = np.argmax(prediction[0])
class_name = class_names[index].strip()
confidence_score = prediction[0][index]

# 8. Render perfect final production evaluation dashboard metrics
print("\n" + "="*30)
print(f"Prediction result: {class_name}")
print(f"Confidence Score: {float(confidence_score) * 100:.2f}%")
print("="*30)


Prediction result: 1 dog
Confidence Score: 93.12%
